# Блокнот 18. Трансформеры в зрении: ViT, сопоставление DETR, RT-DETR и SAM

Курс «Машинное обучение с нуля», модуль 18. Разделы: 1 — урок 18.1, 2 — урок 18.3, 3 — урок 18.4, 4 — урок 18.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

В Colab включите видеокарту: «Среда выполнения → Сменить среду выполнения → T4 GPU». Без неё всё работает, но ViT в разделе 1 учится 3 эпохи вместо 10.

Блокнот скачивает Fashion-MNIST (≈30 МБ), картинку COCO (≈170 КБ) и веса YOLO26n (≈5 МБ), RT-DETR-L (≈66 МБ) и SAM 2.1-tiny (≈78 МБ).

In [ ]:
%pip install -q ultralytics

In [ ]:
import os
import time
import urllib.request
from itertools import permutations

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from scipy.optimize import linear_sum_assignment
from torch import nn
from torchvision import datasets, transforms
from torchvision.ops import box_convert, generalized_box_iou

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("COURSE_DATA", "data")
os.makedirs(DATA, exist_ok=True)
print("устройство:", device, "| torch", torch.__version__)


def draw(ax, boxes, labels=None, color="lime"):
    for k, (x1, y1, x2, y2) in enumerate(boxes):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=2))
        if labels is not None:
            ax.text(x1, y1 - 3, labels[k], color="white", fontsize=8, bbox=dict(fc=color, ec="none", pad=1))

## 1. Маленький ViT своими руками (урок 18.1)

Картинки Fashion-MNIST — 28 × 28, один канал. Нарежем их на патчи 4 × 4: получится 7 · 7 = 49 патчей по 16 чисел.

**Задание 1.** Допишите `patchify(x, p)`: тензор картинок `(B, C, H, W)` → тензор патчей `(B, N, C·p·p)`, патчи в порядке чтения (слева направо, сверху вниз). Подсказка: `x.unfold(2, p, p).unfold(3, p, p)` даёт форму `(B, C, H/p, W/p, p, p)`; переставьте оси (`permute`) так, чтобы впереди шли номера патчей, и сожмите (`reshape`).

In [ ]:
def patchify(x, p):
    B, C, H, W = x.shape
    ...  # ваш код


img = torch.arange(16.0).reshape(1, 1, 4, 4)       # числа 0…15 по строкам
P = patchify(img, 2)
print(P)
assert P.shape == (1, 4, 4)
assert P[0, 0].tolist() == [0, 1, 4, 5] and P[0, 1].tolist() == [2, 3, 6, 7] and P[0, 3].tolist() == [10, 11, 14, 15]
assert patchify(torch.zeros(2, 3, 28, 28), 4).shape == (2, 49, 48)

**Задание 2.** Допишите `forward` маленького ViT: патчи → линейный слой (эмбеддинги) → токен CLS впереди → плюс позиционные эмбеддинги → энкодер трансформера → выход CLS → голова. Токен CLS — один обучаемый вектор на всю модель: размножьте его на батч через `self.cls.expand(B, -1, -1)` и приклейте `torch.cat(..., dim=1)`.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, img=28, p=4, ch=1, dim=64, depth=4, heads=4, classes=10):
        super().__init__()
        self.p = p
        n = (img // p) ** 2
        self.embed = nn.Linear(ch * p * p, dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(dim, heads, dim_feedforward=2 * dim, dropout=0.0, batch_first=True,
                                           norm_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, x):
        B = x.shape[0]
        ...  # ваш код


vit = TinyViT().to(device)
n_params = sum(p.numel() for p in vit.parameters())
print(f"параметров: {n_params:,}".replace(",", " "))
assert vit(torch.zeros(2, 1, 28, 28, device=device)).shape == (2, 10)

Обучим ViT на Fashion-MNIST теми же настройками, что CNN в блокноте 13 (AdamW, шаг 0,001, батч 128). Та маленькая CNN за 10 эпох дала 0,918, а этот ViT за 10 эпох набирает около 0,87 (за 3 эпохи на процессоре — около 0,85): без больших данных ViT уступает свёрточной сети (урок 18.1).

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
train = datasets.FashionMNIST(DATA, train=True, download=True, transform=tf)
test = datasets.FashionMNIST(DATA, train=False, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=1000)


def evaluate(model, loader):
    model.eval()
    correct = n = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
            n += len(y)
    return correct / n


EPOCHS = 10 if device == "cuda" else 3
opt = torch.optim.AdamW(vit.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(EPOCHS):
    vit.train()
    t0 = time.time()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        loss_fn(vit(X), y).backward()
        opt.step()
    print(f"эпоха {epoch + 1}: тест {evaluate(vit, test_loader):.4f} ({time.time() - t0:.0f} с)")
acc = evaluate(vit, test_loader)
assert acc > 0.80
print(f"ViT: {acc:.3f} против 0,918 у маленькой CNN из блокнота 13")

Посмотрим, чему научились позиционные эмбеддинги: для четырёх патчей (красный крестик) — сходство их позиционного вектора с векторами всех 49 позиций, чем светлее, тем похожее. Никто не говорил сети, что патчи лежат на сетке 7 × 7, но соседние позиции постепенно получают похожие векторы. После 3 эпох картина ещё шумная, после 10 — заметнее.

In [ ]:
pos = vit.pos[0, 1:].detach().cpu()
pos = pos / pos.norm(dim=1, keepdim=True)
sim = (pos @ pos.T).reshape(49, 7, 7)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, k in zip(axes, (0, 10, 24, 48)):
    ax.imshow(sim[k], cmap="viridis")
    r, c = divmod(k, 7)
    ax.plot(c, r, "r+", ms=12)
    ax.set_title(f"патч {k}\nстрока {r}, столбец {c}", fontsize=10)
    ax.axis("off")
plt.show()

## 2. Сопоставление предсказаний и объектов, как в DETR (урок 18.3)

Сначала сверим перебор всех перестановок с `scipy.optimize.linear_sum_assignment` — функцией, которая решает задачу о назначениях быстро.

In [ ]:
def best_assignment(cost):
    n = len(cost)
    best, best_total = None, None
    for perm in permutations(range(n)):
        total = sum(cost[i][perm[i]] for i in range(n))
        if best_total is None or total < best_total:
            best, best_total = perm, total
    return best, best_total


rng = np.random.default_rng(0)
for n in (3, 5, 7):
    C = rng.random((n, n))
    t0 = time.time()
    perm, total = best_assignment(C)
    t_brute = time.time() - t0
    rows, cols = linear_sum_assignment(C)
    assert np.isclose(C[rows, cols].sum(), total)
    print(f"n = {n}: перебор {t_brute * 1000:8.1f} мс, сумма {total:.3f} — scipy даёт ту же сумму")

C = rng.random((300, 300))
t0 = time.time()
linear_sum_assignment(C)
print(f"n = 300: scipy за {(time.time() - t0) * 1000:.0f} мс; перебору понадобилось бы 300! вариантов")

Теперь — как в DETR. 10 предсказаний (вероятности 3 классов и рамки) и 3 настоящих объекта. Стоимость пары «предсказание — объект» в DETR:

стоимость = −(вероятность класса объекта) + 5 · L1(рамки в формате cx, cy, w, h) − 2 · GIoU(рамки).

L1 — сумма модулей разностей четырёх чисел рамки, GIoU — обобщённый IoU (как IoU, но для непересекающихся рамок становится отрицательным и показывает, насколько они далеко).

**Задание 3.** Допишите `matching_cost(probs, pred_boxes, gt_classes, gt_boxes)`: верните матрицу стоимостей `(10, 3)`. Подсказки: `probs[:, gt_classes]` — вероятности нужных классов для всех пар сразу; `torch.cdist(a, b, p=1)` — L1 для всех пар; `generalized_box_iou` принимает рамки xyxy, переводите их `box_convert(..., "cxcywh", "xyxy")`.

In [ ]:
def matching_cost(probs, pred_boxes, gt_classes, gt_boxes):
    ...  # ваш код


g = torch.Generator().manual_seed(1)
gt_boxes = torch.tensor([[0.25, 0.5, 0.4, 0.8], [0.75, 0.35, 0.4, 0.6], [0.2, 0.15, 0.2, 0.08]])  # cx, cy, w, h
gt_classes = torch.tensor([0, 0, 1])                                                               # кот, кот, пульт
pred_boxes = torch.rand(10, 4, generator=g) * torch.tensor([1, 1, 0.3, 0.3]) + torch.tensor([0, 0, 0.05, 0.05])
pred_boxes[[2, 5, 7]] = gt_boxes[[1, 2, 0]] + 0.02 * torch.randn(3, 4, generator=g)  # три хороших предсказания
pred_boxes[8] = gt_boxes[0] + 0.15 * torch.randn(4, generator=g)                    # и одно похуже — дубль кота
probs = torch.softmax(torch.randn(10, 3, generator=g), dim=1)
probs[[2, 7, 8]] = torch.tensor([0.9, 0.05, 0.05])
probs[5] = torch.tensor([0.05, 0.9, 0.05])

cost = matching_cost(probs, pred_boxes, gt_classes, gt_boxes)
assert cost.shape == (10, 3)
rows, cols = linear_sum_assignment(cost.numpy())
for r, c in zip(rows, cols):
    print(f"объект {c} ({['кот', 'пульт'][int(gt_classes[c])]}) ← предсказание {r}, стоимость {cost[r, c]:.2f}")
assert dict(zip(cols.tolist(), rows.tolist())) == {0: 7, 1: 2, 2: 5}
print("дубль кота (предсказание 8) остался без пары — его научат отвечать «нет объекта»")

## 3. RT-DETR и YOLO26 на одной картинке (урок 18.4)

Обе модели есть в Ultralytics. Сравним рамки и время на вашем железе — ровно то, что советует урок: мерить у себя, а не верить таблицам.

In [ ]:
from ultralytics import RTDETR, YOLO

img_path = os.path.join(DATA, "coco_000000039769.jpg")
if not os.path.exists(img_path):
    urllib.request.urlretrieve("http://images.cocodataset.org/val2017/000000039769.jpg", img_path)

models = {"YOLO26n": YOLO("yolo26n.pt"), "YOLO26l": YOLO("yolo26l.pt"), "RT-DETR-L": RTDETR("rtdetr-l.pt")}
results = {}
for name, m in models.items():
    m.predict(img_path, verbose=False)  # прогрев
    times = []
    for _ in range(3):
        r = m.predict(img_path, conf=0.5, verbose=False)[0]
        times.append(r.speed["inference"])
    results[name] = r
    n_par = sum(p.numel() for p in m.model.parameters()) / 1e6
    found = [r.names[int(c)] for c in r.boxes.cls]
    print(f"{name:10s} {n_par:5.1f} млн параметров, {np.median(times):6.0f} мс: {found}")
    assert "cat" in found

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, r) in zip(axes, results.items()):
    ax.imshow(r.plot()[..., ::-1])
    ax.set_title(name)
    ax.axis("off")
plt.show()

Все три модели нашли обеих кошек и пульт; большие — ещё второй пульт и диван. Цена — время: на 6-ядерном процессоре автора курса YOLO26n тратит около 40 мс, YOLO26l — около 225 мс, RT-DETR-L — около 380 мс на картинку. Внимание и большая основа RT-DETR дорого обходятся без видеокарты. В Colab с T4 запустите ячейку ещё раз и сравните.

## 4. SAM 2: маски по рамкам детектора (урок 18.5)

SAM не знает классов, а YOLO знает. Возьмём рамки кошек от YOLO26n и попросим SAM 2.1 выделить маски внутри них.

In [ ]:
from ultralytics import SAM

sam = SAM("sam2.1_t.pt")
det = results["YOLO26n"]
cat_boxes = det.boxes.xyxy[det.boxes.cls == 15].cpu()  # класс 15 в COCO — cat
assert len(cat_boxes) == 2
t0 = time.time()
seg = sam(img_path, bboxes=cat_boxes.tolist(), verbose=False)[0]
print(f"SAM 2.1: {len(seg.masks)} маски за {time.time() - t0:.1f} с")
masks = seg.masks.data.cpu().numpy().astype(bool)
for k, (m, b) in enumerate(zip(masks, cat_boxes)):
    box_area = float((b[2] - b[0]) * (b[3] - b[1]))
    print(f"кошка {k}: маска {m.sum()} пикселей — {m.sum() / box_area:.0%} площади рамки")

image = plt.imread(img_path)
overlay = image.copy() / 255.0
for m, color in zip(masks, ([1, 0.4, 0], [0, 0.6, 1])):
    overlay[m] = 0.5 * overlay[m] + 0.5 * np.array(color)
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(overlay)
draw(ax, cat_boxes.numpy(), ["кошка"] * len(cat_boxes), "tab:green")
ax.set_title("рамки YOLO26n → маски SAM 2.1")
ax.axis("off")
plt.show()

Маска занимает лишь 40–60 % рамки: остальное — диван. Вот почему для подсчёта площади или вырезания объекта рамки мало.

**Задание 4.** Допишите `mask_iou_dice(a, b)` для двух булевых масок NumPy (урок 18.5). Затем попросим SAM выделить кошку не рамкой, а **одной точкой** и сравним маски по точке с масками по рамке. Точек две: центр рамки правой кошки и задняя лапа левой.

In [ ]:
def mask_iou_dice(a, b):
    ...  # ваш код


assert np.allclose(mask_iou_dice(np.array([[1, 1, 0]], bool), np.array([[0, 1, 1]], bool)), (1 / 3, 0.5))
assert mask_iou_dice(np.zeros((2, 2), bool), np.zeros((2, 2), bool)) == (1.0, 1.0)

right = int(np.argmax(cat_boxes[:, 0]))       # правая кошка — у неё рамка правее
left = 1 - right
x1, y1, x2, y2 = cat_boxes[right].tolist()
prompts = {"центр правой кошки": ([(x1 + x2) / 2, (y1 + y2) / 2], right),
           "лапа левой кошки": ([60, 400], left)}  # x, y в пикселях картинки 640 × 480

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (title, (point, k)) in zip(axes, prompts.items()):
    m = sam(img_path, points=[point], labels=[1], verbose=False)[0].masks.data[0].cpu().numpy().astype(bool)
    iou, dice = mask_iou_dice(m, masks[k])
    print(f"{title}: маска {m.sum()} пикселей, IoU с маской по рамке {iou:.2f}, Dice {dice:.2f}")
    ax.imshow(image)
    ax.imshow(np.ma.masked_where(~m, m), alpha=0.55, cmap="autumn")
    ax.plot(*point, "c*", ms=15)
    ax.set_title(f"точка: {title}\nIoU с маской по рамке {iou:.2f}")
    ax.axis("off")
plt.show()

Точка в центре кошки дала почти ту же маску, что и рамка, а точка на лапе — только лапу. Для SAM это не ошибка: одна точка — неоднозначная подсказка (урок 18.5), и «лапа» — тоже разумный ответ. Рамка однозначнее. Попробуйте другие точки и несколько точек сразу: `points=[[x1, y1], [x2, y2]], labels=[1, 1]`; метка 0 означает «это **не** объект».